# 6. Parse PDFs With GROBID

This notebook converts downloaded PDFs from `raw_data/publications_<year>` into parsed text files in `parsed_texts/publications_<year>`. It is the first step after PDF download.

Start GROBID before running the parser, for example:

```bash
docker run -d --init -p 8070:8070 grobid/grobid:0.8.2-crf
```


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

import os
import re
import time
import threading
import unicodedata
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# =========================================================
# CONFIG
# =========================================================
# Code is run from the OneDrive project code folder.
CODE_ROOT = REPO_ROOT
DATASET_CURATION_CODE_ROOT = CODE_ROOT / "dataset_curation"

# Data is read from and written to the share-drive project folder.
SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.

YEAR = 2025
pdf_folder = DATA_ROOT / "raw_data" / f"publications_{YEAR}"
output_folder = DATA_ROOT / "parsed_texts" / f"publications_{YEAR}"
status_csv = output_folder / "grobid_processing_status.csv"

grobid_url = os.environ.get("GROBID_URL", "http://localhost:8070")
grobid_endpoint = f"{grobid_url}/api/processFulltextDocument"

max_workers = 4
batch_size = 100
timeout = 30
max_retries = 2
retry_wait = 5

request_data = {
    "flavor": "article/light",
    "consolidateHeader": "0",
    "consolidateCitations": "0",
    "includeRawAffiliations": "0",
    "includeRawCitations": "0",
    "segmentSentences": "0",
    "teiCoordinates": "0"
}

# =========================================================
# METHODS EXTRACTION CONFIG
# =========================================================
fallback_to_fulltext_if_no_methods = False   # True = save fulltext (without references) if no methods found
                                             # False = save only methods, otherwise save empty text

# =========================================================
# RESUME / RETRY CONFIG
# =========================================================
retry_failed = False   # False = continue where status file left off, including old failures
                       # True  = retry rows where success == 0

# =========================================================
# RUN LIMITS
# =========================================================
max_pdfs_this_run = None      # e.g. 5 for testing, or None
max_batches_this_run = None    # e.g. 1, 5, 10, or None

# =========================================================
# SETUP
# =========================================================
output_folder.mkdir(parents=True, exist_ok=True)

# =========================================================
# HELPERS
# =========================================================
def safe_name(filename):
    return re.sub(r"[^\w\-.]", "_", filename)

def save_text(path, text):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text if text else "")

_thread_local = threading.local()

def get_requests_session():
    session = getattr(_thread_local, "session", None)
    if session is None:
        session = requests.Session()
        _thread_local.session = session
    return session

def server_is_up(url):
    try:
        r = get_requests_session().get(url, timeout=10)
        return r.status_code < 500
    except Exception:
        return False

def load_status_df(status_csv):
    expected_cols = [
        "filename",
        "pdf_path",
        "success",
        "seconds",
        "chars_saved",
        "references_cut",
        "methods_found",
        "saved_mode",
        "error"
    ]

    if status_csv.exists():
        df = pd.read_csv(status_csv)
        for col in expected_cols:
            if col not in df.columns:
                df[col] = None
        return df[expected_cols]
    else:
        return pd.DataFrame(columns=expected_cols)

def get_already_processed(status_df, retry_failed=False):
    if status_df.empty:
        return set()

    status_df = status_df.dropna(subset=["filename"]).copy()
    status_df["filename"] = status_df["filename"].astype(str)

    if retry_failed:
        success_series = pd.to_numeric(status_df["success"], errors="coerce").fillna(0)
        return set(status_df.loc[success_series == 1, "filename"].tolist())
    else:
        return set(status_df["filename"].tolist())

# =========================================================
# GROBID REQUEST
# =========================================================
def grobid_request_one(pdf_path, request_data, timeout=300, max_retries=5, retry_wait=8):
    start = time.time()
    last_error = None
    session = get_requests_session()

    for attempt in range(1, max_retries + 1):
        try:
            with open(pdf_path, "rb") as f:
                files = {
                    "input": (os.path.basename(pdf_path), f, "application/pdf")
                }

                response = session.post(
                    grobid_endpoint,
                    files=files,
                    data=request_data,
                    timeout=timeout
                )

            if response.status_code == 503:
                last_error = "503 service busy"
                if attempt < max_retries:
                    time.sleep(retry_wait)
                    continue
                raise RuntimeError(last_error)

            response.raise_for_status()
            runtime = time.time() - start
            return response.text, runtime, None

        except Exception as e:
            last_error = str(e)
            if attempt < max_retries:
                time.sleep(retry_wait)
            else:
                runtime = time.time() - start
                return None, runtime, last_error

# =========================================================
# TEI PARSING
# =========================================================
TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}

REFERENCE_PATTERNS = [
    r"^references$",
    r"^bibliography$",
    r"^bibliografia$",
    r"^works cited$",
    r"^literature cited$",
    r"^reference list$",
    r"^cited literature$",
    r"^referencias$",
    r"^referencias bibliograficas$",
    r"^referencias bibliograficas y notas$",
    r"^referencias? bibliograficas$",
    r"^referencias?$",
    r"^referencias y notas$",
    r"^referenser$",
    r"^literatura$",
    r"^literaturverzeichnis$",
]

SECTION_BOUNDARY_PATTERNS = [
    r"^results?\b",
    r"^resultados\b",
    r"^resultat\b",
    r"^eredmeny",
    r"^discussion\b",
    r"^discusion\b",
    r"^discussao\b",
    r"^diskussion\b",
    r"^conclusions?\b",
    r"^conclusion(?:es)?\b",
    r"^conclusao\b",
    r"^summary\b",
    r"^sammanfattning\b",
    r"^acknowledg?ments?\b",
    r"^agradecimientos\b",
    r"^agradecimento",
    r"^koszonetnyilvanitas",
    r"^author contributions?\b",
    r"^contributions statement\b",
    r"^conflict of interest\b",
    r"^competing interests?\b",
    r"^declaration of interests?\b",
    r"^declarations? of interest\b",
    r"^funding\b",
    r"^permit conditions\b",
    r"^supplement",
    r"^appendix",
    r"^wider implications\b",
    r"^figure legends?\b",
]

METHODS_PATTERNS = [
    r"^methods?\b",
    r"^materials?\s*(?:and|&)\s*methods?\b",
    r"^methods?\s*(?:and|&)\s*materials?\b",
    r"^materials?\b.*\bmethods?\b",
    r"^material(?:es)?\s+y\s+metodos?\b",
    r"^materiais?\s+e\s+metodos?\b",
    r"^metodos?\b",
    r"^metoder\b",
    r"^metody\b",
    r"^methodology\b",
    r"^metodika\b",
    r"^methodik\b",
    r"^experimental methods?\b",
    r"^experimental procedures?\b",
    r"^experimental materials?\s*(?:and|&)\s*methods?\b",
    r"^experimental material\s*(?:and|&)\s*methods?\b",
    r"^experimental design\b",
    r"^experimental model and subject details\b",
    r"^experimental$",
    r"^research methods?\b",
    r"^research design\b",
    r"^data\s*(?:and|&)\s*methods?\b",
    r"^patients?\s*(?:and|&)\s*methods?\b",
    r"^study design\b",
    r"^study area(?:s)?\s+and\s+methods?\b",
    r"^background\s+and\s+methods?\b",
    r"^sampling design\b",
    r"^methoden\b",
    r"^anyag\s+es\s+modszer\b",
    r"^experimental section\b",
    r"^model description\b",
    r"^model formulation\b",
    r"^mathematical model\b",
    r"^numerical simulations?\b",
    r"^observational data\b",
    r"^study area\b",
    r"^area de estudio\b",
    r"^area de estudo\b",
]

METHOD_LIKE_KEYWORDS = [
    "method",
    "methodology",
    "material",
    "metodo",
    "metoder",
    "sampling",
    "sample",
    "study site",
    "study sites",
    "study area",
    "study areas",
    "study location",
    "study locations",
    "collection",
    "collecting",
    "trapping",
    "assay",
    "analysis",
    "analyses",
    "statistical",
    "design",
    "experimental",
    "protocol",
    "questionnaire",
    "survey",
    "pipeline",
    "construction",
    "calibration",
    "testing",
    "filtration",
    "extraction",
    "library preparation",
    "sequencing",
    "demultiplex",
    "adapter",
    "phylogenetic",
    "population genetic",
    "microcosm",
    "monitoring programme",
    "monitoring program",
    "field communities",
    "restoration treatments",
    "data preprocessing",
    "parameter determination",
    "requirements and validation",
    "validation data",
    "uncertainty assessment",
    "governing equations",
    "initial and boundary conditions",
    "model domain",
    "model inputs",
    "area de estudio",
    "area de estudo",
]

METHOD_LIKE_EXCLUDE_PATTERNS = [
    r"\bsupplement",
    r"\bfigure",
    r"\blegend",
    r"\btable\b",
    r"\breference",
    r"\bbibliograph",
    r"\bliterature\b",
    r"\backnowledg",
    r"\bauthor contributions?\b",
    r"\bdata availability\b",
    r"\bdeclaration of interests?\b",
    r"\bcompeting interests?\b",
    r"\bresults?\b.*\bdiscussion\b",
    r"\bdiscussion\b",
]

REFERENCE_REGEXES = [re.compile(pattern) for pattern in REFERENCE_PATTERNS]
SECTION_BOUNDARY_REGEXES = [re.compile(pattern) for pattern in SECTION_BOUNDARY_PATTERNS]
METHODS_REGEXES = [re.compile(pattern) for pattern in METHODS_PATTERNS]
METHOD_LIKE_EXCLUDE_REGEXES = [re.compile(pattern) for pattern in METHOD_LIKE_EXCLUDE_PATTERNS]
METHOD_LIKE_KEYWORD_REGEXES = [
    re.compile(r"\b" + re.escape(keyword) + r"\b")
    for keyword in METHOD_LIKE_KEYWORDS
]
INTRO_HEADING_REGEX = re.compile(
    r"^(introduction|background|introduccion|introducao|inledning|bevezetes)\b"
)
NOISE_NUMBER_REGEX = re.compile(r"^\d+(?:/\d+)?$")
NOISE_EQUATION_REGEX = re.compile(r"^eq\.?\s*\d+$")
NOISE_SYMBOL_REGEX = re.compile(r"^[^a-z]{1,6}$")
INLINE_METHOD_REGEX = re.compile(
    r"\bmaterials?\s*(?:and|&)\s*methods?\b|\bmaterial and methods\b|\bmethods?\b(?::|\s+-|\s+)",
    flags=re.IGNORECASE
)

def normalize_heading(text):
    if not text:
        return ""
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    return " ".join(text.split()).strip().lower()

def heading_matches(text, patterns):
    return heading_matches_normalized(normalize_heading(text), patterns)

def heading_matches_normalized(normalized_heading, patterns):
    for pat in patterns:
        if pat.match(normalized_heading):
            return True
    return False

def is_reference_heading(head):
    return is_reference_heading_normalized(normalize_heading(head))

def is_reference_heading_normalized(normalized_heading):
    return heading_matches_normalized(normalized_heading, REFERENCE_REGEXES)

def is_section_boundary_heading(head):
    return is_section_boundary_heading_normalized(normalize_heading(head))

def is_section_boundary_heading_normalized(normalized_heading):
    return (
        is_reference_heading_normalized(normalized_heading)
        or heading_matches_normalized(normalized_heading, SECTION_BOUNDARY_REGEXES)
    )

def is_methods_heading(head):
    return is_methods_heading_normalized(normalize_heading(head))

def is_methods_heading_normalized(normalized_heading):
    return heading_matches_normalized(normalized_heading, METHODS_REGEXES)

def is_intro_heading(head):
    return is_intro_heading_normalized(normalize_heading(head))

def is_intro_heading_normalized(normalized_heading):
    return bool(INTRO_HEADING_REGEX.match(normalized_heading))

def is_noise_heading(head):
    return is_noise_heading_normalized(normalize_heading(head))

def is_noise_heading_normalized(normalized_heading):
    if not normalized_heading:
        return True
    if NOISE_NUMBER_REGEX.match(normalized_heading):
        return True
    if NOISE_EQUATION_REGEX.match(normalized_heading):
        return True
    if NOISE_SYMBOL_REGEX.match(normalized_heading):
        return True
    return False

def is_method_like_heading(head):
    return is_method_like_heading_normalized(normalize_heading(head))

def is_method_like_heading_normalized(normalized_heading):
    if not normalized_heading:
        return False
    for pat in METHOD_LIKE_EXCLUDE_REGEXES:
        if pat.search(normalized_heading):
            return False
    for keyword_regex in METHOD_LIKE_KEYWORD_REGEXES:
        if keyword_regex.search(normalized_heading):
            return True
    return False

def extract_paragraphs_from_div(div, ns, start_regex=None):
    paras = []
    started = start_regex is None

    for p in div.findall(".//tei:p", ns):
        txt = "".join(p.itertext()).strip()
        if not txt:
            continue

        if not started:
            match = start_regex.search(txt)
            if not match:
                continue
            txt = txt[match.start():].strip()
            started = True

        paras.append(txt)
    return paras

def get_heading(div, ns):
    head_el = div.find("tei:head", ns)
    if head_el is None:
        return ""
    return "".join(head_el.itertext()).strip()

def trim_paragraphs_from_match(paras, start_regex):
    trimmed = []
    started = False

    for txt in paras:
        if not started:
            match = start_regex.search(txt)
            if not match:
                continue
            txt = txt[match.start():].strip()
            started = True

        if txt:
            trimmed.append(txt)

    return trimmed

def build_body_records(body, ns):
    records = []

    for elem in list(body):
        tag = elem.tag.split("}")[-1]

        if tag == "div":
            heading = get_heading(elem, ns)
            records.append({
                "tag": "div",
                "heading": heading,
                "heading_norm": normalize_heading(heading),
                "paras": extract_paragraphs_from_div(elem, ns),
            })
        elif tag == "p":
            txt = "".join(elem.itertext()).strip()
            records.append({
                "tag": "p",
                "text": txt,
            })

    return records

def extract_fulltext_before_references(tei_xml):
    if isinstance(tei_xml, str):
        root = ET.fromstring(tei_xml.encode("utf-8"))
    else:
        root = ET.fromstring(tei_xml)

    body = root.find(".//tei:text/tei:body", TEI_NS)
    if body is None:
        paragraphs = []
        for p in root.findall(".//tei:text//tei:p", TEI_NS):
            txt = "".join(p.itertext()).strip()
            if txt:
                paragraphs.append(txt)
        return "\n\n".join(paragraphs).strip(), False

    records = build_body_records(body, TEI_NS)
    collected_parts = []
    references_cut = False

    for record in records:
        if record["tag"] == "div":
            if is_reference_heading_normalized(record["heading_norm"]):
                references_cut = True
                break

            paras = record["paras"]
            if paras:
                collected_parts.append("\n\n".join(paras))
        elif record["tag"] == "p" and record["text"]:
            collected_parts.append(record["text"])

    text = "\n\n".join(part for part in collected_parts if part).strip()
    return text, references_cut

def collect_methods_block(records, start_index, inline_start_regex=None):
    collected_parts = []
    started = False

    for idx, record in enumerate(records):
        if idx < start_index:
            continue

        if record["tag"] == "div":
            if idx > start_index and is_section_boundary_heading_normalized(record["heading_norm"]):
                break

            paras = record["paras"]
            if idx == start_index and inline_start_regex is not None:
                paras = trim_paragraphs_from_match(paras, inline_start_regex)

            if paras:
                collected_parts.append("\n\n".join(paras))
                started = True
            elif started and idx > start_index:
                continue
        elif record["tag"] == "p" and (started or idx == start_index):
            txt = record["text"]
            if txt:
                collected_parts.append(txt)
                started = True

    return "\n\n".join(part for part in collected_parts if part).strip()

def extract_methods_section(tei_xml):
    if isinstance(tei_xml, str):
        root = ET.fromstring(tei_xml.encode("utf-8"))
    else:
        root = ET.fromstring(tei_xml)

    body = root.find(".//tei:text/tei:body", TEI_NS)
    if body is None:
        return "", False

    records = build_body_records(body, TEI_NS)

    for idx, record in enumerate(records):
        if record["tag"] != "div":
            continue

        if is_methods_heading_normalized(record["heading_norm"]):
            text = collect_methods_block(records, idx)
            if text:
                return text, True

    for idx, record in enumerate(records):
        if record["tag"] != "div":
            continue

        if is_intro_heading_normalized(record["heading_norm"]) or not record["heading_norm"]:
            paras = record["paras"]
            for para in paras:
                if INLINE_METHOD_REGEX.search(para):
                    text = collect_methods_block(records, idx, inline_start_regex=INLINE_METHOD_REGEX)
                    if text:
                        return text, True
                    break

    collected_parts = []
    active = False
    non_method_streak = 0
    saw_signal = False

    for record in records:
        if record["tag"] != "div":
            continue

        if is_section_boundary_heading_normalized(record["heading_norm"]):
            if active:
                break
            continue

        if is_intro_heading_normalized(record["heading_norm"]):
            continue

        if is_noise_heading_normalized(record["heading_norm"]):
            if active:
                continue
            continue

        is_signal = is_method_like_heading_normalized(record["heading_norm"])

        if not active:
            if not is_signal:
                continue
            active = True
            saw_signal = True

        paras = record["paras"]
        if paras:
            collected_parts.append("\n\n".join(paras))

        if is_signal:
            non_method_streak = 0
        else:
            non_method_streak += 1
            if non_method_streak >= 2:
                break

    text = "\n\n".join(part for part in collected_parts if part).strip()
    if saw_signal and text:
        return text, True

    return "", False

def extract_text_to_save(tei_xml, fallback_to_fulltext_if_no_methods=True):
    methods_text, methods_found = extract_methods_section(tei_xml)

    if methods_found and methods_text.strip():
        return methods_text, True, False, "methods"

    if fallback_to_fulltext_if_no_methods:
        full_text, references_cut = extract_fulltext_before_references(tei_xml)
        return full_text, False, references_cut, "fulltext_fallback"

    return "", False, False, "none"

# =========================================================
# WORKER
# =========================================================
def process_one_pdf(pdf_path):
    tei_xml, runtime, error = grobid_request_one(
        pdf_path=pdf_path,
        request_data=request_data,
        timeout=timeout,
        max_retries=max_retries,
        retry_wait=retry_wait
    )

    row = {
        "filename": pdf_path.name,
        "pdf_path": str(pdf_path),
        "success": 0,
        "seconds": round(runtime, 4),
        "chars_saved": None,
        "references_cut": False,
        "methods_found": False,
        "saved_mode": None,
        "error": error
    }

    if error is not None:
        return row

    try:
        final_text, methods_found, references_cut, saved_mode = extract_text_to_save(
            tei_xml,
            fallback_to_fulltext_if_no_methods=fallback_to_fulltext_if_no_methods
        )

        stem = safe_name(pdf_path.stem)
        txt_path = output_folder / f"{stem}.txt"

        has_text_to_save = bool(final_text and final_text.strip())
        if has_text_to_save:
            save_text(txt_path, final_text)
        else:
            saved_mode = "none"

        row["success"] = 1
        row["chars_saved"] = len(final_text) if has_text_to_save else 0
        row["references_cut"] = references_cut
        row["methods_found"] = methods_found
        row["saved_mode"] = saved_mode
        row["error"] = None

    except Exception as e:
        row["success"] = 0
        row["error"] = str(e)

    return row


# =========================================================
# MAIN
# =========================================================
print("Checking GROBID server...")
if not server_is_up(grobid_url):
    raise RuntimeError(
        f"GROBID not reachable at {grobid_url}. "
        f"Start it first, e.g.:\n"
        f"docker run -d --init -p 8070:8070 grobid/grobid:0.8.2-crf"
    )

status_df = load_status_df(status_csv)
already_processed = get_already_processed(status_df, retry_failed=retry_failed)
already_processed_before_run = len(already_processed)

pdf_files = sorted([p for p in pdf_folder.iterdir() if p.suffix.lower() == ".pdf"])
remaining_files_all = [p for p in pdf_files if p.name not in already_processed]

if max_pdfs_this_run is not None:
    remaining_files = remaining_files_all[:max_pdfs_this_run]
else:
    remaining_files = remaining_files_all

print(f"Total PDFs found: {len(pdf_files)}")
print(f"Already processed before this run: {already_processed_before_run}")
print(f"Remaining PDFs available: {len(remaining_files_all)}")
print(f"PDFs selected for this run: {len(remaining_files)}")
print(f"Retry failed rows: {retry_failed}")

rows_added_this_run = 0

# =========================================================
# BATCH PROCESSING
# =========================================================
if len(remaining_files) > 0:
    total_batches_available = (len(remaining_files) + batch_size - 1) // batch_size

    if max_batches_this_run is not None:
        batches_to_run = min(total_batches_available, max_batches_this_run)
    else:
        batches_to_run = total_batches_available

    print(f"Batches available from current remaining files: {total_batches_available}")
    print(f"Batches scheduled for this run: {batches_to_run}")

    for batch_index in range(batches_to_run):
        batch_start = batch_index * batch_size
        batch_files = remaining_files[batch_start:batch_start + batch_size]
        batch_nr = batch_index + 1

        batch_t0 = time.time()
        batch_rows = []

        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futures = {ex.submit(process_one_pdf, pdf_path): pdf_path for pdf_path in batch_files}

            for fut in as_completed(futures):
                row = fut.result()
                batch_rows.append(row)

        batch_df = pd.DataFrame(batch_rows)
        rows_added_this_run += len(batch_df)

        status_df = pd.concat([status_df, batch_df], ignore_index=True)
        status_df.to_csv(status_csv, index=False, encoding="utf-8-sig")

        batch_time = time.time() - batch_t0
        batch_success = int(pd.to_numeric(batch_df["success"], errors="coerce").fillna(0).sum())
        batch_failed = len(batch_df) - batch_success
        batch_refs_cut = int(pd.to_numeric(batch_df["references_cut"], errors="coerce").fillna(0).sum())
        batch_methods_found = int(pd.to_numeric(batch_df["methods_found"], errors="coerce").fillna(0).sum())

        print(
            f"Finished batch {batch_nr}/{batches_to_run} | "
            f"size={len(batch_df)} | success={batch_success} | failed={batch_failed} | "
            f"methods_found={batch_methods_found} | refs_cut={batch_refs_cut} | time={batch_time:.2f}s"
        )

else:
    print("Nothing left to do.")

# =========================================================
# FINAL SUMMARY
# =========================================================
status_df = pd.read_csv(status_csv)

success_series = pd.to_numeric(status_df["success"], errors="coerce").fillna(0)
success_df = status_df[success_series == 1].copy()

run_df = status_df.tail(rows_added_this_run).copy() if rows_added_this_run > 0 else pd.DataFrame(columns=status_df.columns)
run_success_series = pd.to_numeric(run_df["success"], errors="coerce").fillna(0) if len(run_df) > 0 else pd.Series(dtype=float)
run_success_df = run_df[run_success_series == 1].copy() if len(run_df) > 0 else pd.DataFrame(columns=status_df.columns)

print("\n================ FINAL SUMMARY ================\n")

print("This run")
print(f"  rows added this run: {rows_added_this_run}")
print(f"  successful this run: {len(run_success_df)}")
print(f"  failed this run: {len(run_df) - len(run_success_df)}")
if len(run_success_df) > 0:
    print(f"  mean runtime this run: {run_success_df['seconds'].mean():.3f} s")
    print(f"  median runtime this run: {run_success_df['seconds'].median():.3f} s")
    print(f"  mean chars saved this run: {run_success_df['chars_saved'].mean():.0f}")
    print(f"  methods found this run: {int(pd.to_numeric(run_success_df['methods_found'], errors='coerce').fillna(0).sum())}/{len(run_success_df)}")
    print(f"  references cut this run: {int(pd.to_numeric(run_success_df['references_cut'], errors='coerce').fillna(0).sum())}/{len(run_success_df)}")

print("\nOverall status file")
print(f"  processed total overall: {len(status_df)}")
print(f"  successful total overall: {len(success_df)}")
print(f"  failed total overall: {len(status_df) - len(success_df)}")
if len(success_df) > 0:
    print(f"  mean runtime overall: {success_df['seconds'].mean():.3f} s")
    print(f"  median runtime overall: {success_df['seconds'].median():.3f} s")
    print(f"  mean chars saved overall: {success_df['chars_saved'].mean():.0f}")
    print(f"  methods found overall: {int(pd.to_numeric(success_df['methods_found'], errors='coerce').fillna(0).sum())}/{len(success_df)}")
    print(f"  references cut overall: {int(pd.to_numeric(success_df['references_cut'], errors='coerce').fillna(0).sum())}/{len(success_df)}")